In [7]:
import csv
import datetime
from pathlib import Path
from collections import defaultdict
import pandas as pd

# ==========================================
# 1. CONFIGURATION
# ==========================================

MY_NETID = "cdamron3"
TEAMMATE_NETID = "ncash3"

STATS_FILE = "ncash3_project_stats.csv"
SUMMARY_FILE = "ncash3_project_summary.csv"

OUTPUT_FILE = f"{MY_NETID}_validation_log.csv"

FIELDS = [
    "HasRecovered",
    "WhyRecovered",
    "ActivityPattern",
    "Currentstatus",
    "RecentThemes"
]

# ==========================================
# 2. READ TEAMMATE'S DATA
# ==========================================

# The original files use semicolons as separators.
with open(STATS_FILE, newline="", encoding="utf-8-sig") as f:
    stats = list(csv.DictReader(f, delimiter=";"))

with open(SUMMARY_FILE, newline="", encoding="utf-8-sig") as f:
    commit_rows = list(csv.reader(f, delimiter=";"))

# Organize commits by project
commits_by_project = defaultdict(list)

for row in commit_rows:
    if len(row) < 5:
        continue

    try:
        date = datetime.datetime.fromtimestamp(
            int(row[3]), datetime.timezone.utc
        ).date()
    except (ValueError, OverflowError):
        continue

    project = row[0]
    message = row[4].splitlines()[0].strip()

    commits_by_project[project].append((date, message))

for project in commits_by_project:
    commits_by_project[project].sort(reverse=True)

# ==========================================
# 3. MY INDEPENDENT CLASSIFICATIONS
# ==========================================

# These are the judgments we discussed.
# You can edit any value here.

judgments = {
    "afeinstein20_eleanor": {
        "HasRecovered": "Yes",
        "WhyRecovered": "New TESS fields, faster cadences, and Sector 65 data needed support",
        "ActivityPattern": "Declining",
        "Currentstatus": "Active",
        "RecentThemes": "Bug Fixes, Merges"
    },

    "algebraicjulia_catlab.jl": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Normal development resumed with new notebooks and cleanup",
        "ActivityPattern": "Declining",
        "Currentstatus": "Active",
        "RecentThemes": "Dependency Updates, Merges"
    },

    "microprediction_timemachines": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Maintainer returned to fix CI workflows, refresh docs, and update mds",
        "ActivityPattern": "Declining",
        "Currentstatus": "Active",
        "RecentThemes": "Documentation Updates, Other"
    },

    "ntustison_crosslong": {
        "HasRecovered": "No",
        "WhyRecovered": "N/A - Never recovered",
        "ActivityPattern": "Stopped",
        "Currentstatus": "Active",
        "RecentThemes": "Documentation Updates"
    },

    "patwie_cppnumericalsolvers": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Major rewrite: Bazel 8, Nelder-Mead, constrained optimization, faster L-BFGS",
        "ActivityPattern": "Steady",
        "Currentstatus": "Active",
        "RecentThemes": "Bug Fixes, Dependency Updates"
    },

    "sftcd_surveys": {
        "HasRecovered": "Yes",
        "WhyRecovered": "New contributors extended the scripts (MaxMind GeoIP tooling)",
        "ActivityPattern": "Irregular",
        "Currentstatus": "Active",
        "RecentThemes": "Dependency Updates, Other"
    },

    "simplenlg_simplenlg": {
        "HasRecovered": "Yes",
        "WhyRecovered": "Fixes for newer JDKs and numbered issue reports (#54, #65, #75, #76)",
        "ActivityPattern": "Declining",
        "Currentstatus": "Inactive",
        "RecentThemes": "Documentation Updates, Dependency Updates"
    },

    "smtorg_smt": {
        "HasRecovered": "N/A - No break in commits",
        "WhyRecovered": "N/A - No break in commits",
        "ActivityPattern": "Rising",
        "Currentstatus": "Active",
        "RecentThemes": "Automated Bot Contributions, Documentation Updates"
    },

    "stan-dev_projpred": {
        "HasRecovered": "Yes",
        "WhyRecovered": "CRAN fixes, news updates and test fixes",
        "ActivityPattern": "Steady",
        "Currentstatus": "Active",
        "RecentThemes": "Documentation Updates, Release"
    },

    "whizzml_examples": {
        "HasRecovered": "Yes",
        "WhyRecovered": "New time series preprocessing utilities",
        "ActivityPattern": "Declining",
        "Currentstatus": "Active",
        "RecentThemes": "Feature Development, Bug Fixes"
    }
}

# ==========================================
# 4. RECORD OUR DISAGREEMENTS
# ==========================================

# These are the classifications you changed.
# All other project-field combinations are agreements.

disagreements = {
    "WhyRecovered": {
        "microprediction_timemachines"
    },

    "ActivityPattern": {
        "ntustison_crosslong",
        "patwie_cppnumericalsolvers",
        "simplenlg_simplenlg",
        "smtorg_smt",
        "stan-dev_projpred"
    },

    "Currentstatus": {
        "simplenlg_simplenlg"
    },

    "RecentThemes": {
        "afeinstein20_eleanor",
        "algebraicjulia_catlab.jl",
        "ntustison_crosslong",
        "patwie_cppnumericalsolvers"
    }
}

# ==========================================
# 5. GENERATE SUPPORTING EVIDENCE
# ==========================================

REFERENCE_DATE = datetime.date(2025, 9, 30)
ACTIVE_START = datetime.date(2025, 4, 1)

def get_evidence(project_stats, field):
    project = project_stats["Project"]

    commits = [
        (date, message)
        for date, message in commits_by_project[project]
        if date <= REFERENCE_DATE
    ]

    recent = commits[:10]

    if field in ["HasRecovered", "WhyRecovered"]:
        return (
            f"Longest gap: {project_stats['LongestGapStart']} "
            f"to {project_stats['LongestGapEnd']} "
            f"({project_stats['LongestGapLength']} months). "
            f"Commits after gap: {project_stats['NumCommitsAfterLastGap']}. "
            f"Before themes: {project_stats['BeforeThemes']}. "
            f"After themes: {project_stats['AfterThemes']}. "
            f"Recovery evidence: {project_stats['WhyRecovered']}."
        )

    elif field == "ActivityPattern":
        return (
            f"Total commits: {project_stats['ncommits']}. "
            f"Timeline: {project_stats['from']} to {project_stats['to']}. "
            f"Number of gaps: {project_stats['NumberOfGapsInTimeline']}. "
            f"Original notes: {project_stats['Notes'][:200]}"
        )

    elif field == "Currentstatus":
        active_commits = [
            date for date, message in commits
            if ACTIVE_START <= date <= REFERENCE_DATE
        ]

        latest = commits[0][0] if commits else "None"

        return (
            f"Commits between April 1 and September 30, 2025: "
            f"{len(active_commits)}. "
            f"Latest recorded commit: {latest}."
        )

    elif field == "RecentThemes":
        subjects = [
            f"{date}: {message[:100]}"
            for date, message in recent[:5]
        ]

        return "Recent commit messages: " + " | ".join(subjects)

    return ""

# ==========================================
# 6. CREATE THE 50 VALIDATION ROWS
# ==========================================

columns = [
    "WOCProjectID",
    "ProjectOwnerNetID",
    "ValidatorNetID",
    "Field",
    "AuthorValue",
    "ValidatorValue",
    "PreDiscussAgreementstatus(Yes/No)",
    "AdjudicatedValue",
    "KeyEvidence",
    "Comments"
]

validation_rows = []

# Field-first ordering, as required by Part 1.
for field in FIELDS:

    for project_stats in stats:
        project = project_stats["Project"]

        author_value = project_stats[field].strip()
        validator_value = judgments[project][field]

        agreement = (
            "No"
            if project in disagreements.get(field, set())
            else "Yes"
        )

        validation_rows.append({
            "WOCProjectID": project,
            "ProjectOwnerNetID": TEAMMATE_NETID,
            "ValidatorNetID": MY_NETID,
            "Field": field,
            "AuthorValue": author_value,
            "ValidatorValue": validator_value,
            "PreDiscussAgreementstatus(Yes/No)": agreement,
            "AdjudicatedValue": "",
            "KeyEvidence": get_evidence(project_stats, field),
            "Comments": ""
        })

# ==========================================
# 7. SAVE THE CSV
# ==========================================

df = pd.DataFrame(validation_rows, columns=columns)

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"Created: {OUTPUT_FILE}")
print(f"Total rows: {len(df)}")
print(f"Projects: {df['WOCProjectID'].nunique()}")
print(f"Fields: {df['Field'].nunique()}")

display(df)


Created: cdamron3_validation_log.csv
Total rows: 50
Projects: 10
Fields: 5


,WOCProjectID,ProjectOwnerNetID,ValidatorNetID,Field,AuthorValue,ValidatorValue,PreDiscussAgreementstatus(Yes/No),AdjudicatedValue,KeyEvidence,Comments
0,afeinstein20_eleanor,ncash3,cdamron3,HasRecovered,Yes,Yes,Yes,,Longest gap: 2023-06 to 2024-04 (11 months). C...,
1,algebraicjulia_catlab.jl,ncash3,cdamron3,HasRecovered,Yes,Yes,Yes,,Longest gap: 2019-04 to 2019-06 (3 months). Co...,
2,microprediction_timemachines,ncash3,cdamron3,HasRecovered,Yes,Yes,Yes,,Longest gap: 2023-07 to 2024-09 (15 months). C...,
3,ntustison_crosslong,ncash3,cdamron3,HasRecovered,No,No,Yes,,Longest gap: 2022-01 to 2025-06 (42 months). C...,
4,patwie_cppnumericalsolvers,ncash3,cdamron3,HasRecovered,Yes,Yes,Yes,,Longest gap: 2022-09 to 2023-12 (16 months). C...,
5,sftcd_surveys,ncash3,cdamron3,HasRecovered,Yes,Yes,Yes,,Longest gap: 2019-08 to 2021-08 (25 months). C...,
6,simplenlg_simplenlg,ncash3,cdamron3,HasRecovered,Yes,Yes,Yes,,Longest gap: 2018-09 to 2020-02 (18 months). C...,
7,smtorg_smt,ncash3,cdamron3,HasRecovered,N/A,N/A - No break in commits,Yes,,Longest gap: 2022-10 to 2022-11 (2 months). Co...,
8,stan-dev_projpred,ncash3,cdamron3,HasRecovered,Yes,Yes,Yes,,Longest gap: 2018-12 to 2019-02 (3 months). Co...,
9,whizzml_examples,ncash3,cdamron3,HasRecovered,Yes,Yes,Yes,,Longest gap: 2024-02 to 2025-03 (14 months). C...,


In [8]:
# Confirm exactly 50 rows
assert len(df) == 50

# Confirm each field has 10 projects
assert (df.groupby("Field").size() == 10).all()

# Confirm adjudication columns are empty
assert (df["AdjudicatedValue"] == "").all()
assert (df["Comments"] == "").all()

# Show agreement counts
print("Agreement Summary:")
print(df["PreDiscussAgreementstatus(Yes/No)"].value_counts())

# Show disagreements
display(
    df.loc[
        df["PreDiscussAgreementstatus(Yes/No)"] == "No",
        [
            "WOCProjectID",
            "Field",
            "AuthorValue",
            "ValidatorValue"
        ]
    ]
)


Agreement Summary:
PreDiscussAgreementstatus(Yes/No)
Yes    39
No     11
Name: count, dtype: int64


,WOCProjectID,Field,AuthorValue,ValidatorValue
12,microprediction_timemachines,WhyRecovered,Maintainer returned to fix CI workflows and re...,"Maintainer returned to fix CI workflows, refre..."
23,ntustison_crosslong,ActivityPattern,declining,Stopped
24,patwie_cppnumericalsolvers,ActivityPattern,U-shaped,Steady
26,simplenlg_simplenlg,ActivityPattern,irregular,Declining
27,smtorg_smt,ActivityPattern,steady,Rising
28,stan-dev_projpred,ActivityPattern,declining,Steady
36,simplenlg_simplenlg,Currentstatus,Inactive,Inactive
40,afeinstein20_eleanor,RecentThemes,Bug fixes: Other,"Bug Fixes, Merges"
41,algebraicjulia_catlab.jl,RecentThemes,Dependency updates:Other,"Dependency Updates, Merges"
43,ntustison_crosslong,RecentThemes,Other,Documentation Updates
